# FANUC-style conveyor pick-and-place

Reproducible kinematic baseline for a six-axis M-10iD/12-inspired arm. The link geometry is an explicitly approximate proxy, not FANUC CAD or a dynamics-validated digital twin. This kernel records the Kaggle GPU/driver, runs focused package checks, and then attempts the pinned Isaac Lab 3.0 OvPhysX smoke test when the runtime driver meets the documented CUDA 13 minimum.


In [ ]:
from __future__ import annotations
import json, os, platform, re, subprocess, sys, time
from pathlib import Path

working_dir = Path('/kaggle/working')
working_dir.mkdir(exist_ok=True)
report_path = working_dir / 'runtime-report.json'
report = {'commands': []}
RUN_BACKEND_SMOKES = False
stage_started = {}
def stage_start(name):
    stage_started[name] = time.perf_counter()
def stage_end(name):
    elapsed = round(time.perf_counter() - stage_started.pop(name), 3)
    timings = report.setdefault('timings_seconds', {})
    timings[name] = elapsed
    print(f'[PERF] stage={name} seconds={elapsed}')
    latest = globals().get('runtime_report')
    if isinstance(latest, dict):
        latest.setdefault('timings_seconds', {}).update(timings)
    save_report()
def save_report():
    latest = globals().get('runtime_report')
    if isinstance(latest, dict):
        report.update(latest)
    report_path.write_text(json.dumps(report, indent=2))

def run(command, **kwargs):
    started = time.perf_counter()
    result = subprocess.run(command, text=True, capture_output=True, check=False, **kwargs)
    elapsed = round(time.perf_counter() - started, 3)
    report.setdefault('commands', []).append({'argv': command, 'exit_code': result.returncode,
        'duration_seconds': elapsed,
        'stdout_tail': result.stdout[-8000:], 'stderr_tail': result.stderr[-8000:]})
    print(f'[PERF] command_seconds={elapsed} exit={result.returncode} command={command[0]}')
    save_report()
    return result

stage_start('gpu_preflight')
gpu = run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'])
driver = run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'])
report.update({'python': sys.version, 'platform': platform.platform(), 'gpu_query_exit': gpu.returncode,
          'gpus': [line.strip() for line in gpu.stdout.splitlines() if line.strip()],
          'driver_versions': sorted(set(line.strip() for line in driver.stdout.splitlines() if line.strip()))})
report['gpu_count'] = len(report['gpus'])
report['kaggle_t4_detected'] = any('t4' in line.lower() for line in report['gpus'])
print(json.dumps(report, indent=2))
save_report()
if gpu.returncode != 0 or not report['gpus']:
    raise RuntimeError('Kaggle session has no visible NVIDIA GPU; select GPU accelerator.')
caps = []
try:
    import torch
    report['torch'] = torch.__version__
    for device in range(torch.cuda.device_count()):
        caps.append(torch.cuda.get_device_capability(device))
    report['torch_cuda_devices'] = torch.cuda.device_count()
    report['compute_capabilities'] = [list(cap) for cap in caps]
except Exception as exc:
    report['torch_probe_error'] = repr(exc)
driver_text = report['driver_versions'][0] if report['driver_versions'] else '0'
match = re.match(r'(\d+)\.(\d+)(?:\.(\d+))?', driver_text)
driver_tuple = tuple(map(int, match.groups(default='0'))) if match else (0, 0, 0)
report['isaaclab_linux_driver_compatible'] = driver_tuple >= (580, 65, 6)
report['isaaclab_gpu_arch_compatible'] = bool(caps) and all(cap >= (7, 5) for cap in caps)
Path('/kaggle/working').mkdir(exist_ok=True)
Path('/kaggle/working/runtime-report.json').write_text(json.dumps(report, indent=2))
stage_end('gpu_preflight')


In [ ]:
import base64, io, zipfile
import importlib.util, os, shutil
from pathlib import Path
project_zip_b64 = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(project_zip_b64))) as archive:
    archive.extractall('/kaggle/working/fanuc-cell')
%cd /kaggle/working/fanuc-cell
stage_start('local_setup')
project_src = str(Path.cwd() / 'src')
os.environ['PYTHONPATH'] = project_src + os.pathsep + os.environ.get('PYTHONPATH', '')
if importlib.util.find_spec('pytest') is None:
    pytest_install = run([sys.executable, '-m', 'pip', 'install', 'pytest>=8'])
    if pytest_install.returncode != 0:
        raise RuntimeError('Could not install pytest: ' + pytest_install.stderr[-4000:])
stage_end('local_setup')


In [ ]:
import json, subprocess
from pathlib import Path
stage_start('local_tests')
test_env = os.environ.copy()
test_env['PYTHONPATH'] = str(Path.cwd() / 'src') + os.pathsep + test_env.get('PYTHONPATH', '')
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True, env=test_env)
stage_end('local_tests')


## Isaac Lab custom task

On a compatible T4 runtime, this installs Isaac Lab 3.0.0, runs the official kit-less PhysX smoke, generates the official external manager-based PPO project, imports the six-axis arm URDF into USD, and steps the conveyor task with a zero-action agent before training PPO for 500 iterations. It then plays the checkpoint for one episode while recording a headless headless video. The report keeps startup, conversion, training, playback, and episode evidence distinct; video alone is not a success-rate claim.


In [ ]:
import shutil
from pathlib import Path
runtime_report = json.loads(Path('/kaggle/working/runtime-report.json').read_text())
if runtime_report['isaaclab_gpu_arch_compatible'] and runtime_report['isaaclab_linux_driver_compatible']:
    uv_probe = run(['uv', '--version'])
    if uv_probe.returncode != 0:
        install_uv = run([sys.executable, '-m', 'pip', 'install', 'uv'])
        if install_uv.returncode != 0:
            raise RuntimeError(install_uv.stderr[-4000:])
    source_dir = Path('/kaggle/working/IsaacLab')
    clone = run(['git', 'clone', '--depth', '1', '--branch', 'v3.0.0',
                 'https://github.com/isaac-sim/IsaacLab.git', str(source_dir)])
    if clone.returncode != 0:
        raise RuntimeError('Could not clone Isaac Lab v3.0.0: ' + clone.stderr[-4000:])
    sync = run(['uv', 'sync'], cwd=source_dir)
    if sync.returncode != 0:
        raise RuntimeError('Isaac Lab dependency installation failed: ' + sync.stderr[-6000:])
    if RUN_BACKEND_SMOKES:
        smoke = run(['uv', 'run', '--extra', 'ovphysx', 'isaaclab', 'zero_agent', '--task', 'Isaac-Cartpole-Direct',
                     '--num_envs', '1', '--max_steps', '64', '--viz', 'none',
                     'physics=ovphysx'], cwd=source_dir)
        runtime_report['isaaclab_smoke_exit'] = smoke.returncode
        runtime_report['isaaclab_smoke_stdout'] = smoke.stdout[-6000:]
        runtime_report['isaaclab_smoke_stderr'] = smoke.stderr[-6000:]
        print(smoke.stdout[-6000:])
        if smoke.returncode != 0:
            raise RuntimeError('Isaac Lab OvPhysX smoke test failed; see runtime report.')
    else:
        runtime_report['isaaclab_smoke_exit'] = None
        runtime_report['isaaclab_smoke_skip'] = 'Previously validated; full custom baseline runs below.'
    project_dir = Path('/kaggle/working/fanuc_cell_lab')
    generate = run(['uv', 'run', 'isaaclab', '--new', '--non_interactive',
                   '--task_type', 'external', '--project_path', '/kaggle/working',
                   '--name', 'fanuc_cell_lab', '--author', 'Pascal Coimbra',
                   '--task_name', 'conveyor_pick_place', '--robot_name', 'm10id12',
                   '--workflow', 'manager-based:single-agent',
                   '--rl_library', 'rsl_rl', '--rl_algorithm', 'ppo'], cwd=source_dir)
    if generate.returncode != 0 or not project_dir.is_dir():
        raise RuntimeError('Isaac Lab project generation failed: ' + generate.stderr[-6000:])
    overlay = Path('/kaggle/working/fanuc-cell/isaaclab_overlay/src/fanuc_cell_lab')
    project_package = project_dir / 'src' / 'fanuc_cell_lab'
    shutil.copytree(overlay, project_package, dirs_exist_ok=True)
    shutil.copytree('/kaggle/working/fanuc-cell/src/fanuc_cell',
                    project_dir / 'src' / 'fanuc_cell', dirs_exist_ok=True)
    shutil.copy2('/kaggle/working/fanuc-cell/scripts/run_scripted_demo.py',
                 project_dir / 'scripts' / 'run_scripted_demo.py')
    shutil.copy2('/kaggle/working/fanuc-cell/scripts/diagnose_training_nan.py',
                 project_dir / 'scripts' / 'diagnose_training_nan.py')
    asset_dir = project_package / 'assets' / 'data'
    asset_dir.mkdir(parents=True, exist_ok=True)
    converted_asset = asset_dir / 'm10id12_robot' / 'm10id12_robot.usd'
    official_converter = (source_dir / 'scripts/tools/convert_urdf.py').read_text()
    converted_source, replacements = re.subn(
        r'(args_cli\.physics\s*=\s*\"isaacsim_physx\"\s+if\s+args_cli\.require_kit\s+else\s+)\"[^\"]+\"',
        r'\1"ovphysx"', official_converter, count=1)
    if replacements != 1:
        raise RuntimeError('Pinned Isaac Lab URDF converter changed its backend selection; inspect before conversion.')
    converter_script = source_dir / 'scripts/tools/convert_urdf_ovphysx.py'
    converter_script.write_text(converted_source)
    conversion = run(['uv', 'run', '--extra', 'importers', '--extra', 'ovphysx', 'isaaclab', '-p',
                     str(converter_script),
                     '/kaggle/working/fanuc-cell/assets/m10id12_robot.urdf',
                     str(asset_dir), '--joint-stiffness', '0.0',
                     '--joint-damping', '0.0', '--joint-target-type', 'none'], cwd=source_dir)
    runtime_report['urdf_conversion_exit'] = conversion.returncode
    runtime_report['urdf_conversion_stdout'] = conversion.stdout[-6000:]
    runtime_report['urdf_conversion_stderr'] = conversion.stderr[-6000:]
    if conversion.returncode != 0 or not converted_asset.is_file():
        candidates = sorted(asset_dir.rglob('m10id12_robot.usd*'))
        if conversion.returncode != 0 or not candidates:
            raise RuntimeError('Isaac Lab URDF-to-USD conversion failed: ' + conversion.stderr[-6000:])
        source_usd = candidates[0]
        converted_asset.parent.mkdir(parents=True, exist_ok=True)
        if source_usd.resolve() != converted_asset.resolve():
            shutil.copy2(source_usd, converted_asset)
    contact_code = (
        'import json,sys; from pxr import Usd,UsdPhysics,Sdf; '
        'stage=Usd.Stage.Open(sys.argv[1]); '
        'bodies=[p for p in stage.Traverse() if p.HasAPI(UsdPhysics.RigidBodyAPI)]; '
        "[(p.AddAppliedSchema('PhysxRigidBodyAPI'), p.AddAppliedSchema('PhysxContactReportAPI'), "
        " p.CreateAttribute('physxRigidBody:sleepThreshold', Sdf.ValueTypeNames.Float, custom=False).Set(0.0), "
        " p.CreateAttribute('physxContactReport:threshold', Sdf.ValueTypeNames.Float, custom=False).Set(0.0)) "
        " for p in bodies]; "
        "stage.GetRootLayer().Save(); print(json.dumps({'contact_report_bodies':len(bodies)}))"
    )
    contact_patch = run(['uv', 'run', 'python', '-c', contact_code, str(converted_asset)], cwd=source_dir)
    if contact_patch.returncode != 0:
        raise RuntimeError('Failed to enable contact reporting in converted robot USD: ' + contact_patch.stderr[-4000:])
    runtime_report['usd_contact_patch'] = json.loads(contact_patch.stdout.strip().splitlines()[-1])
    if runtime_report['usd_contact_patch']['contact_report_bodies'] < 1:
        raise AssertionError('No rigid bodies found while enabling USD contact reporting.')
    inspect_code = (
        'import json,sys; from pxr import Usd,UsdPhysics; '
        'stage=Usd.Stage.Open(sys.argv[1]); ps=list(stage.Traverse()); '
        'r=sum(p.IsA(UsdPhysics.RevoluteJoint) for p in ps); '
        'j=sum(p.IsA(UsdPhysics.PrismaticJoint) for p in ps); '
        'b=[str(p.GetPath()) for p in ps if p.HasAPI(UsdPhysics.RigidBodyAPI)]; '
        'print(json.dumps({\"revolute_joints\":int(r),\"prismatic_joints\":int(j),\"rigid_bodies\":b}))'
    )
    inspect = run(['uv', 'run', 'python', '-c', inspect_code, str(converted_asset)], cwd=source_dir)
    if inspect.returncode != 0:
        raise RuntimeError('USD articulation inspection failed: ' + inspect.stderr[-4000:])
    runtime_report['converted_usd'] = str(converted_asset)
    usd_summary = json.loads(inspect.stdout.strip().splitlines()[-1])
    runtime_report['usd_joint_counts'] = {key: usd_summary[key] for key in ('revolute_joints', 'prismatic_joints')}
    runtime_report['usd_rigid_bodies'] = usd_summary['rigid_bodies']
    if runtime_report['usd_joint_counts'] != {'revolute_joints': 6, 'prismatic_joints': 2}:
        raise AssertionError('Converted articulation must contain 6 revolute + 2 gripper slide joints: ' + str(runtime_report['usd_joint_counts']))
    if not any('tool0' in path.lower() for path in runtime_report['usd_rigid_bodies']):
        raise AssertionError('Converted USD is missing the tool0 rigid body: ' + str(runtime_report['usd_rigid_bodies']))
    baseline_env = os.environ.copy()
    baseline_env['PYTHONPATH'] = str(project_dir / 'src') + os.pathsep + baseline_env.get('PYTHONPATH', '')
    baseline_env['FANUC_CELL_PHYSICS'] = 'ovphysx'
    nan_probe = run(['uv', 'run', '--extra', 'ovphysx', 'python',
                    str(project_dir / 'scripts' / 'diagnose_training_nan.py'),
                    '--device', 'cuda:0', '--num-envs', '64', '--steps', '24', '--seed', '7'],
                   cwd=source_dir, env=baseline_env)
    runtime_report['ppo_nan_probe_exit'] = nan_probe.returncode
    runtime_report['ppo_nan_probe_stdout'] = nan_probe.stdout[-8000:]
    runtime_report['ppo_nan_probe_stderr'] = nan_probe.stderr[-4000:]
    print(nan_probe.stdout[-8000:])
    if nan_probe.returncode != 0:
        save_report()
        raise RuntimeError('PPO environment probe found non-finite observations. stdout:\n' + nan_probe.stdout[-8000:] + '\nstderr:\n' + nan_probe.stderr[-4000:])
    baseline = run(['uv', 'run', '--extra', 'ovphysx', 'python',
                    str(project_dir / 'scripts' / 'run_scripted_demo.py'),
                    '--device', 'cuda:0', '--seed', '23',
                    '--output', '/kaggle/working/isaaclab-baseline.json'],
                   cwd=source_dir, env=baseline_env)
    runtime_report['scripted_baseline_exit'] = baseline.returncode
    runtime_report['scripted_baseline_stdout'] = baseline.stdout[-8000:]
    runtime_report['scripted_baseline_stderr'] = baseline.stderr[-8000:]
    baseline_metrics = Path('/kaggle/working/isaaclab-baseline.json')
    if baseline_metrics.is_file():
        runtime_report['scripted_baseline'] = json.loads(baseline_metrics.read_text())
    if baseline.returncode != 0 or not baseline_metrics.is_file():
        save_report()
        raise RuntimeError('Scripted physical baseline did not complete. stdout:\n' + baseline.stdout[-12000:] + '\nstderr head:\n' + baseline.stderr[:8000] + '\nstderr tail:\n' + baseline.stderr[-2000:])
    baseline_result = runtime_report['scripted_baseline']
    if not baseline_result.get('success') or not baseline_result.get('scripted_grasp_sensor_passed'):
        save_report()
        raise RuntimeError('Isaac Lab physical baseline failed grasp/place acceptance; PPO training is gated until it passes: ' + str(baseline_result))
    task_sync = run(['uv', 'sync'], cwd=project_dir)
    if task_sync.returncode != 0:
        raise RuntimeError('Custom Isaac Lab project installation failed: ' + task_sync.stderr[-6000:])
    task_query = run(['uv', 'run', 'python', '-c',
                      'import json, gymnasium as gym; import fanuc_cell_lab.tasks; '
                      'print(json.dumps([k for k in gym.registry if \"conveyor\" in k.lower() and \"m10id12\" in k.lower()]))'],
                     cwd=project_dir)
    if task_query.returncode != 0:
        raise RuntimeError('Custom task registration failed: ' + task_query.stderr[-6000:])
    task_ids = json.loads(task_query.stdout.strip().splitlines()[-1])
    if len(task_ids) != 1:
        raise RuntimeError('Expected one registered conveyor task, got: ' + str(task_ids))
    task_id = task_ids[0]
    runtime_report['cell_task_id'] = task_id
    training = run(['uv', 'run', '--extra', 'ovphysx', 'isaaclab', 'train', '--rl_library', 'rsl_rl',
                    '--task', task_id, '--num_envs', '64', '--max_iterations', '500',
                    '--seed', '7', 'physics=ovphysx'], cwd=project_dir)
    runtime_report['ppo_training_exit'] = training.returncode
    runtime_report['ppo_training_stdout'] = training.stdout[-12000:]
    runtime_report['ppo_training_stderr'] = training.stderr[-12000:]
    checkpoints = sorted(project_dir.glob('logs/rsl_rl/**/*.pt'), key=lambda p: p.stat().st_mtime)
    runtime_report['ppo_checkpoints'] = [{'path': str(p), 'bytes': p.stat().st_size} for p in checkpoints[-3:]]
    if training.returncode != 0:
        save_report()
        raise RuntimeError('RSL-RL PPO training failed. stdout:\n' + training.stdout[-8000:] + '\nstderr:\n' + training.stderr[-12000:])
    if not checkpoints:
        raise RuntimeError('PPO exited without producing an RSL-RL checkpoint.')
    playback = run(['uv', 'run', '--extra', 'ovphysx', '--extra', 'video', 'isaaclab', 'play',
                    '--rl_library', 'rsl_rl', '--task', task_id,
                    '--checkpoint', 'latest', '--num_envs', '1', '--seed', '107',
                    '--viz', 'newton_gl', '--video', '--video_length', '400',
                    'physics=ovphysx'], cwd=project_dir)
    runtime_report['policy_playback_exit'] = playback.returncode
    runtime_report['policy_playback_stdout'] = playback.stdout[-8000:]
    runtime_report['policy_playback_stderr'] = playback.stderr[-8000:]
    if playback.returncode != 0:
        save_report()
        raise RuntimeError('Trained-policy playback/video failed: ' + playback.stderr[-8000:])
    policy_videos = sorted(project_dir.glob('logs/rsl_rl/**/videos/**/*.mp4'),
                           key=lambda p: p.stat().st_mtime)
    if not policy_videos:
        save_report()
        raise RuntimeError('Trained-policy playback completed without an MP4 artifact.')
    policy_video = Path('/kaggle/working/fanuc-cell-isaaclab-policy.mp4')
    shutil.copy2(policy_videos[-1], policy_video)
    runtime_report['policy_video'] = str(policy_video)
    runtime_report['policy_video_bytes'] = policy_video.stat().st_size
    evaluation_dir = Path('/kaggle/working/policy-evaluation')
    evaluation = run(['uv', 'run', '--extra', 'ovphysx', 'isaaclab', 'benchmark', 'play',
                      '--rl_library', 'rsl_rl', '--task', task_id,
                      '--checkpoint', 'latest', '--num_envs', '16',
                      '--num_steps', '800', '--seed', '101',
                      '--viz', 'none', '--benchmark_formatter', 'schema,summary',
                      '--output_path', str(evaluation_dir), 'physics=ovphysx'],
                     cwd=project_dir)
    runtime_report['policy_evaluation_exit'] = evaluation.returncode
    runtime_report['policy_evaluation_stdout'] = evaluation.stdout[-10000:]
    runtime_report['policy_evaluation_stderr'] = evaluation.stderr[-10000:]
    if evaluation.returncode != 0:
        save_report()
        raise RuntimeError('Trained-policy evaluation failed: ' + evaluation.stderr[-10000:])
    evaluation_results = sorted(evaluation_dir.rglob('*.json'), key=lambda p: p.stat().st_mtime)
    if evaluation_results:
        evaluation_data = [json.loads(path.read_text()) for path in evaluation_results]
        runtime_report['policy_evaluation'] = evaluation_data[-1]
        runtime_report['policy_success_rate'] = None
        for _data in evaluation_data:
            for _record in (_data if isinstance(_data, list) else [_data]):
                if not isinstance(_record, dict):
                    continue
                if _record.get('success_rate') is not None:
                    runtime_report['policy_success_rate'] = _record['success_rate']
                for _m in (_record.get('measurements') or []):
                    if isinstance(_m, dict) and str(_m.get('name', '')).endswith('play success_rate'):
                        runtime_report['policy_success_rate'] = _m.get('value')
        if runtime_report['policy_success_rate'] is None:
            save_report()
            raise RuntimeError('Isaac Lab benchmark did not report Metrics/success_rate for completed episodes.')
    else:
        runtime_report['policy_success_rate'] = None
        runtime_report['policy_evaluation_artifact_missing'] = True
        save_report()
        raise RuntimeError('Isaac Lab benchmark did not produce a JSON evaluation result.')
else:
    runtime_report['isaaclab_smoke_exit'] = None
    runtime_report['isaaclab_smoke_skip'] = 'GPU capability or documented Linux driver minimum failed.'
    print(runtime_report['isaaclab_smoke_skip'])
Path('/kaggle/working/runtime-report.json').write_text(json.dumps(runtime_report, indent=2))


In [ ]:
from IPython.display import FileLink, display
for artifact in ['/kaggle/working/fanuc-cell-isaaclab-policy.mp4', '/kaggle/working/runtime-report.json']:
    display(FileLink(artifact))
print(json.loads(Path('/kaggle/working/runtime-report.json').read_text()))
